In [5]:
import sys
import os 

path_to_src = "../src"

if path_to_src not in sys.path:
    sys.path.append(os.path.abspath(path_to_src))
    
from model import *
from utils import *

In [ ]:
def run_all_thresholds(X, true_outliers, informative_idx, run):
    configs = {
        "soft-soft": ("soft", "soft"),
        "soft-SCAD": ("soft", "scad"),
        "SCAD-soft": ("scad", "soft"),
        "SCAD-SCAD": ("scad", "scad"),
    }

    results = {}

    for name, (thE, thW) in configs.items():
        lambda1, lambda2 = select_lambda(X, thresh_E=thE, thresh_w=thW)

        res = arsk(
            X,
            n_clusters=3,
            lambda1=lambda1,
            lambda2=lambda2,
            thresh_E=thE,
            thresh_w=thW,
            random_state=run
        )
        
        n_out, n_feat = evaluate_result(
            res["weights"],
            res["errors"],
            true_outliers,
            informative_idx
        )

        results[name] = (n_out, n_feat)

    return results

In [7]:
def simulation_full(debug=True):
    contamination_levels = [0, 0.1, 0.2, 0.3]
    n_runs = 30

    all_results = {}

    print("=== START SIMULATION ===")
    print(f"Total runs per setting: {n_runs}")
    print(f"Contamination levels: {contamination_levels}\n")

    for pi in contamination_levels:

        print(f"\n===== Contamination π = {pi} =====")

        config_results = {
            "soft-soft": [],
            "soft-SCAD": [],
            "SCAD-soft": [],
            "SCAD-SCAD": []
        }

        for run in range(n_runs):

            if debug:
                print(f"\n--- Run {run+1}/{n_runs} ---")

            X, y, true_outliers, informative_idx = generate_data(
                contamination=pi,
                random_state=run
            )

            res = run_all_thresholds(X, true_outliers, informative_idx, run)

            for k in config_results:
                config_results[k].append(res[k])

                if debug:
                    out, feat = res[k]
                    print(f"{k}: outliers={out}, features={feat}")

        # ===== compute mean/std =====
        summary = {}
        print("\n>>> SUMMARY for π =", pi)

        for k in config_results:
            outs = [x[0] for x in config_results[k]]
            feats = [x[1] for x in config_results[k]]

            summary[k] = {
                "out_mean": np.mean(outs),
                "out_std": np.std(outs),
                "feat_mean": np.mean(feats),
                "feat_std": np.std(feats)
            }

            print(f"\n{k}:")
            print(f"  Outliers = {summary[k]['out_mean']:.2f} ± {summary[k]['out_std']:.2f}")
            print(f"  Features = {summary[k]['feat_mean']:.2f} ± {summary[k]['feat_std']:.2f}")

        all_results[pi] = summary

    print("\n=== END SIMULATION ===")
    return all_results

In [ ]:
results = simulation_full(debug=False)

for pi, configs in results.items():
    print(f"\n" + "="*10 + f" π = {pi} " + "="*10)

    for name, val in configs.items():
        print(f"{name}:")
        
        print(f"  Outliers = {val['out_mean']:.2f} (±{val['out_std']:.2f})")
        print(f"  Features = {val['feat_mean']:.2f} (±{val['feat_std']:.2f})")

print("\n✅ Hoàn thành! Kết quả đã được hiển thị bên trên.")

=== START SIMULATION ===
Total runs per setting: 30
Contamination levels: [0, 0.1, 0.2, 0.3]


===== Contamination π = 0 =====

>>> SUMMARY for π = 0

soft-soft:
  Outliers = 0.00 ± 0.00
  Features = 4.97 ± 0.18

soft-SCAD:
  Outliers = 0.00 ± 0.00
  Features = 4.97 ± 0.18

SCAD-soft:
  Outliers = 0.00 ± 0.00
  Features = 4.97 ± 0.18

SCAD-SCAD:
  Outliers = 0.00 ± 0.00
  Features = 4.97 ± 0.18

===== Contamination π = 0.1 =====

>>> SUMMARY for π = 0.1

soft-soft:
  Outliers = 10.33 ± 4.99
  Features = 5.00 ± 0.00

soft-SCAD:
  Outliers = 10.33 ± 4.99
  Features = 5.00 ± 0.00

SCAD-soft:
  Outliers = 8.33 ± 4.71
  Features = 5.00 ± 0.00

SCAD-SCAD:
  Outliers = 8.33 ± 4.71
  Features = 5.00 ± 0.00

===== Contamination π = 0.2 =====

>>> SUMMARY for π = 0.2

soft-soft:
  Outliers = 20.67 ± 9.98
  Features = 5.00 ± 0.00

soft-SCAD:
  Outliers = 20.67 ± 9.98
  Features = 5.00 ± 0.00

SCAD-soft:
  Outliers = 14.00 ± 8.00
  Features = 5.00 ± 0.00

SCAD-SCAD:
  Outliers = 13.33 ± 7.45
  Fea